#### Lab 02 - Collaborative Filtering (User, Film, Rating)

Maintain User Engagement by Generating Recommendation Using Other User Preference

Tackle High Sparsity Rate by Filtering on User Film Threshold

#### What Will We Perform ?

We'll Pivot the Rating Table

Perform Limitation on Film to User, User to Film to Decrease Sparse Rate

Join Rating and Film Table

Create Model to Perform Recommender

In [1]:
import warnings, pandas

warnings.filterwarnings("ignore")

In [2]:
# Use Rating Table

rating = pandas.read_table("ratings.csv", sep=",")

rating.iloc[:5]

,UserID,MovieID,Rating,Timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [3]:
rating.tail()

,UserID,MovieID,Rating,Timestamp
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352
100835,610,170875,3.0,1493846415


In [4]:
null = rating.isnull().sum()

null.sum()

0

In [5]:
total = rating.shape[0]

total

100836

In [6]:
total_film = rating["MovieID"].nunique()

total_film

9724

In [7]:
total_user = rating["UserID"].nunique()

total_user

610

On Our Rating Table, There are 610 User & 9.724 Film

In [8]:
rata_rata_rating = rating["Rating"].mean()

rata_rata_rating

3.501556983616962

Out of 100.836 User Interaction with Film, Average Rating is 3.5 out of 5.0

Create Pivot to Rating Table and Perform Sparse Rate

In [9]:
# Create Rating Table

rating_table = rating.pivot_table(index="MovieID", columns="UserID", values="Rating")

# Create Sparse Rate Function to Check Sparse Rate

SparseRate = lambda table : round((table.isnull().sum().sum() / table.size) * 100, 3)

SparseRate(rating_table)

98.3

Our 98.3 % Sparsity Rate Signaling Many Things.

* Our Table Mostly Consists of 0

* Our Films Haven't been Watched

* Our Films is Watched, but the User Didn't Rate the Film

* One or Two Films is Watch Less than 10 Times or User Only Watch Less Than 50 Movie

Our Solution is to Set a Minimum Threshold to Film - User Interaction

The First Threshold is Film. Our Minimum Threshold for a Film is

*This Film Must be Watched, Minimum of 10 Times*

In [10]:
minimum_film_interaction = 10

WatchFilm = rating.groupby("MovieID")["Rating"].agg(["count", "mean"])

WatchFilm.columns = ["TotalWatch", "Rating"]

WatchFilm = WatchFilm[WatchFilm["TotalWatch"] > minimum_film_interaction]

rating = rating[rating["MovieID"].isin(WatchFilm.index)]

rating["MovieID"].nunique()

2121

In [11]:
rating_table = rating.pivot_table(index="MovieID", columns="UserID", values="Rating")

SparseRate(rating_table)

93.845

The Last Threshold is User. Our Minimum Threshold for a User is

*This User Must be Watch & Rate, Minimum of 50 Films*

In [12]:
minimum_user_interaction = 50

UserWatch = rating.groupby("UserID")[["Rating"]].count()

UserWatch = UserWatch[UserWatch["Rating"] > minimum_user_interaction]

rating = rating[rating["UserID"].isin(UserWatch.index)]

In [13]:
rating_table = rating.pivot_table(index="MovieID", columns="UserID", values="Rating")

SparseRate(rating_table)

90.396

Our Final Sparsity Rate is 90.39 %

Create Table Film to Join with Rating Table & Perform Feature Engineering

In [14]:
# Use Movies Table on Table Film

table_film = pandas.read_table("movies.csv", sep=",")

table_film = table_film.drop_duplicates("Title")

# Perform Genre Feature Engineering

table_film["Genre"] = table_film["Genre"].apply(lambda i : " ".join(i.split("|")))

table_film = table_film[table_film["Genre"] != "(no genres listed)"]

# Replace Film Genre

replacer = {"Sci-Fi":"SciFi", "Film-Noir":"Filnoir"}

for i, t in replacer.items():

  table_film["Genre"] = table_film["Genre"].str.replace(i, t)

table_film.iloc[:5]

,MovieID,Title,Genre
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy


In [15]:
# Join Rating and Table Film to Further Use

rating = rating.join(table_film.set_index("MovieID"), on="MovieID")

rating = rating.drop("Timestamp", axis="columns")

rating.iloc[:5]

,UserID,MovieID,Rating,Title,Genre
0,1,1,4.0,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,1,3,4.0,Grumpier Old Men (1995),Comedy Romance
2,1,6,4.0,Heat (1995),Action Crime Thriller
3,1,47,5.0,Seven (a.k.a. Se7en) (1995),Mystery Thriller
4,1,50,5.0,"Usual Suspects, The (1995)",Crime Mystery Thriller


In [16]:
rating_table.iloc[:5, :8]

UserID,1,4,6,7,10,11,15,16
MovieID,,,,,,,,
1,4.0,NaN,NaN,4.5,NaN,NaN,2.5,NaN
2,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN
3,4.0,NaN,5.0,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN
6,4.0,NaN,4.0,NaN,NaN,5.0,NaN,NaN


In [17]:
# Fill Missing Value with 0

rating_table = rating_table.fillna(0)

rating_table.iloc[:5, :8]

UserID,1,4,6,7,10,11,15,16
MovieID,,,,,,,,
1,4.0,0.0,0.0,4.5,0.0,0.0,2.5,0.0
2,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0
3,4.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0
6,4.0,0.0,4.0,0.0,0.0,5.0,0.0,0.0


In [18]:
rating_table.index.name

'MovieID'

In [19]:
len(rating_table.index)

2121

In [20]:
rating_table.columns.name

'UserID'

In [21]:
len(rating_table.columns)

352

What is Sparse Matrix ? Sparse Matrix is Matrix that Many of the Value is 0

Usually Sparse Matrix is Mostly Use in Collaborative Method Recommendation System

In [22]:
from scipy.sparse import csr_matrix

rating_matrix = csr_matrix(rating_table.values)

rating_matrix.shape

(2121, 352)

In [23]:
rating_matrix.toarray()[:5, :8]

array([[4. , 0. , 0. , 4.5, 0. , 0. , 2.5, 0. ],
       [0. , 0. , 4. , 0. , 0. , 0. , 0. , 0. ],
       [4. , 0. , 5. , 0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 5. , 0. , 0. , 0. , 0. , 0. ],
       [4. , 0. , 4. , 0. , 0. , 5. , 0. , 0. ]])

In [24]:
rating_matrix.toarray()[0, :10]

array([4. , 0. , 0. , 4.5, 0. , 0. , 2.5, 0. , 4.5, 3.5])

In [25]:
# Reset Rating Table

rating_reset = rating_table.reset_index()

rating_reset.shape

(2121, 353)

In [26]:
rating_reset.iloc[:10, :10]

UserID,MovieID,1,4,6,7,10,11,15,16,17
0,1,4.0,0.0,0.0,4.5,0.0,0.0,2.5,0.0,4.5
1,2,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,4.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0
3,5,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0
4,6,4.0,0.0,4.0,0.0,0.0,5.0,0.0,0.0,0.0
5,7,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0
6,9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,10,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,0.0
8,11,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0
9,12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
rating_reset.columns.name

'UserID'

In [28]:
rating_reset.index.name